# Testing Recursive-Set

In [ ]:
import { RecursiveSet, RecursiveMap, Tuple, Value, hashValue, emptySet, singleton } from './recursive-set';

## Some Helper Functionality

In [ ]:
const SEED : number = 1337;

In [ ]:
function createRNG(seed: number) {
    return function() {
        var t = seed += 0x6D2B79F5;
        t = Math.imul(t ^ (t >>> 15), t | 1);
        t ^= t + Math.imul(t ^ (t >>> 7), t | 61);
        return ((t ^ (t >>> 14)) >>> 0) / 4294967296;
    }
}
const random = createRNG(SEED);

In [ ]:
function measure<T>(label: string, fn: () => T): T {
    const start = performance.now();
    const result = fn();
    const end = performance.now();
    console.log(`[PERF] ${label}: ${(end - start).toFixed(2)}ms`);
    return result;
}

In [ ]:
function assert(condition: boolean, message: string) {
    if (!condition) {
        console.error(`[FAIL] ${message}`);
    } else {
        console.log(`[PASS] ${message}`);
    }
}

## Tests for Recursive-Set and Tuple

In [ ]:
console.log('--- Test 1: Primitive Value Equality ---');
const str1 = "hello";
const str2 = "hel" + "lo"; 

const setStrings = new RecursiveSet<string>();
setStrings.add(str1);

assert(setStrings.has(str2), "Set identifies strings by value");
assert(setStrings.size === 1, "Set prevents duplicate values");

In [ ]:
console.log('--- Test 2: Mixed Types (Primitives vs Sets) ---');
const mixedSet = new RecursiveSet<number | RecursiveSet<number>>();
const innerSet = new RecursiveSet(1);

mixedSet.add(1);          
mixedSet.add(innerSet);   

assert(mixedSet.size === 2, "Set contains both primitive 1 and set {1}");
assert(mixedSet.has(1), "Contains primitive 1");
assert(mixedSet.has(new RecursiveSet(1)), "Contains set {1}");

const it = mixedSet[Symbol.iterator]();
const first = it.next().value;
assert(typeof first === 'number', "Ordering: Primitives must precede Sets");

In [ ]:
console.log('--- Test 3: Deep Structural Equality ---');

const A = new RecursiveSet<Value>(
    new RecursiveSet<Value>(1, 2), 
    new RecursiveSet<Value>(3)
);

const B = new RecursiveSet<Value>(
    new RecursiveSet<Value>(3), 
    new RecursiveSet<Value>(2, 1)
);

assert(A.equals(B), "Sets are equal regardless of insertion order");

In [ ]:
console.log('--- Test 4: Cartesian Product ---');
const setX = new RecursiveSet<Value>(1, 2, "ia", "i", 4);
const setY = new RecursiveSet<Value>(3, 4, "11", "12", "2");
const product = setX.cartesianProduct(setY);

assert(product.size === 25, "Product size is correct");

In [ ]:
console.log('--- Test 5: Immutability of Operations ---');
const base = new RecursiveSet<Value>(1);
const unionResult = base.union(new RecursiveSet<Value>(2));

assert(base.size === 1, "Original set remains unmodified");
assert(unionResult.size === 2, "New set contains result");

In [ ]:
console.log('--- Test 6: Basic Stress Test (1k items) ---');
const startStress = performance.now();
const stressSet = new RecursiveSet<number>();
for(let i=0; i<100000; i++) stressSet.add(i);
assert(stressSet.size === 100000, "Successfully added 100000 items");
console.log(`Took ${(performance.now() - startStress).toFixed(2)}ms`);

In [ ]:
console.log('--- Test 7: Von Neumann Ordinals ---');
type Ordinal = RecursiveSet<Ordinal>;
const zero: Ordinal = new RecursiveSet();
const one: Ordinal = new RecursiveSet(zero);
const two: Ordinal = new RecursiveSet(zero, one);

assert(two.has(one), "2 contains 1");
assert(two.has(zero), "2 contains 0");

In [ ]:
console.log('--- Test 8: Power Set ---');
const baseSet = new RecursiveSet<Value>(1, 3, 11, 25);
const pSet = baseSet.powerset();
assert(pSet.size === 16, "Power set size is correct (16)");
assert(pSet.has(emptySet<number>()), "Contains empty set");

In [ ]:
console.log('--- Test 9: Symmetric Difference ---');
const setA = new RecursiveSet<Value>(1, 2);
const setB = new RecursiveSet<Value>(2, 3);
const symDiff = setA.symmetricDifference(setB);
assert(symDiff.size === 2, "Result size correct {1, 3}");
assert(symDiff.has(1) && symDiff.has(3), "Correct elements");

In [ ]:
console.log('--- Test 10: Recursion Depth ---');
type NestedSet = RecursiveSet<string | NestedSet>;
let current: NestedSet = new RecursiveSet("bottom");
const onionBag = new RecursiveSet<NestedSet>();

for(let i = 0; i < 20; i++) {
    onionBag.add(current);
    const sizeAfterAdd = onionBag.size;
    const expectedSize = i + 1;
    const nextLayer = new RecursiveSet(current);
    
    if (sizeAfterAdd !== expectedSize) {
        console.error("CRITICAL FAILURE: Collision detected");
        break;
    }
    current = nextLayer;
}
assert(onionBag.size === 20, `Expected size 20, got ${onionBag.size}`);

In [ ]:
console.log('--- Test 11: Clone/Copy ---');
const original = new RecursiveSet<Value>(1);
const copy = original.clone();
copy.add(2);
assert(original.size === 1, "Original unmodified");
assert(copy.size === 2, "Copy modified");

In [ ]:
console.log('--- Test 12: Freeze-on-Hash Lifecycle ---');
const lifecycleSet = new RecursiveSet<number>();
lifecycleSet.add(1);
const _hashTrigger = lifecycleSet.hashCode; 

let mutationThrew = false;
try { lifecycleSet.add(3); } catch (e) { mutationThrew = true; }
assert(mutationThrew, "Hashed set throws on mutation (Frozen State)");

In [ ]:
console.log('--- Test 13: Security & Invariant Challenge ---');
const safeArray = [1, 2, 3];
const safeTuple = new Tuple(...safeArray);
safeArray.push(4);
assert(safeTuple.length === 3, "Tuple ignores external array mutation");

const bigIntSet = new RecursiveSet<number>();
bigIntSet.add(-1);
bigIntSet.add(4294967295); 
assert(bigIntSet.size === 2, "Handles 32-bit hash collisions");

const zeroSet = new RecursiveSet<number>();
zeroSet.add(0);
zeroSet.add(-0);
assert(zeroSet.size === 1, "-0 and +0 are same");

In [ ]:
console.log('--- Test 14: Finite Boundaries ---');
const extremeSet = new RecursiveSet<number>();
const MAX = Number.MAX_SAFE_INTEGER;
const MIN = Number.MIN_SAFE_INTEGER;
extremeSet.add(MAX); extremeSet.add(MAX + 1); extremeSet.add(MIN);
assert(extremeSet.size === 3, "Can handle MAX/MIN SAFE INTEGERs");

In [ ]:
console.log('--- Test 15: Property Based Testing ---');
let propsPassed = true;
for(let i=0; i<100; i++) {
    const a = new RecursiveSet(random());
    const b = new RecursiveSet(random());
    if (RecursiveSet.compare(a, a) !== 0) propsPassed = false;
    if (a.equals(b) !== b.equals(a)) propsPassed = false;
}
assert(propsPassed, "Basic Reflexivity & Symmetry passed");

In [ ]:
console.log('--- Test 16: Transitive Freeze ---');
const innerMutable = new RecursiveSet<Value>(1);
const outerMutable = new RecursiveSet(innerMutable);
const _ = outerMutable.hashCode; 
let innerThrew = false;
try { innerMutable.add(2); } catch (e) { innerThrew = true; }
assert(innerThrew, "Outer Set hash freezes Inner Set");

## Tests for Recursive-Map

In [ ]:
console.log('--- Test 1: Standard DFA Transitions (Tuple Keys) ---');
const delta = new RecursiveMap<Tuple<[number, string]>, number>();

const k1 = new Tuple(0, 'a');
const k2 = new Tuple(0, 'b');

delta.set(k1, 1);
delta.set(k2, 2);

assert(delta.size === 2, "Map has 2 transitions");
assert(delta.get(new Tuple(0, 'a')) === 1, "Lookup (0, 'a') -> 1");
assert(delta.get(new Tuple(0, 'b')) === 2, "Lookup (0, 'b') -> 2");
assert(delta.get(new Tuple(0, 'c')) === undefined, "Lookup (0, 'c') -> undefined");

In [ ]:
console.log('--- Test 2: Powerset Construction (Set Keys) ---');
const dfaStates = new RecursiveMap<RecursiveSet<number>, string>();

const q0 = new RecursiveSet(0);
const q0q1 = new RecursiveSet(0, 1);
const qDead = new RecursiveSet<number>(); 

dfaStates.set(q0, "Start");
dfaStates.set(q0q1, "Active");
dfaStates.set(qDead, "Trap");

const lookupActive = new RecursiveSet(1, 0); 
const lookupDead = new RecursiveSet<number>();

assert(dfaStates.get(lookupActive) === "Active", "Set {1,0} matches Key {0,1}");
assert(dfaStates.get(lookupDead) === "Trap", "Empty Set Key works");
assert(dfaStates.get(q0) !== dfaStates.get(lookupActive), "{0} and {0,1} are distinct");

In [ ]:
console.log('--- Test 3: Complex Keys: Tuple([Set], Char) ---');
const transitionTable = new RecursiveMap<Tuple<[RecursiveSet<number>, string]>, RecursiveSet<number>>();

const stateSet = new RecursiveSet(1, 2, 3);
const resultState = new RecursiveSet(4);
const key = new Tuple(stateSet, 'a'); 

transitionTable.set(key, resultState);

const searchKey = new Tuple(new RecursiveSet(3, 1, 2), 'a');
const found = transitionTable.get(searchKey);

assert(found !== undefined, "Complex Lookup found");
assert(found ? found.equals(resultState) : false, "Result Value Structural Equality");

In [ ]:
console.log('--- Test 4: Recursive Map-Inception ---');
const inner = new RecursiveMap<string, number>();
inner.set("a", 1);

const outer = new RecursiveMap<RecursiveMap<string, number>, string>();
outer.set(inner, "Found Inner");

const innerClone = new RecursiveMap<string, number>();
innerClone.set("a", 1);

assert(outer.get(innerClone) === "Found Inner", "Map found via structural clone");

const innerModified = innerClone.mutableCopy();
innerModified.set("b", 2);

assert(outer.get(innerModified) === undefined, "Modified Map is strictly distinct");

In [ ]:
console.log('--- Test 5: Stress Test (100000 items) ---');
measure('Insert & Lookup 100000 Tuples', () => {
    const map = new RecursiveMap<Tuple<[number]>, number>();
    for (let i = 0; i < 100000; i++) {
        map.set(new Tuple(i), i * 2);
    }
    assert(map.size === 100000, "Size is 1000");
    assert(map.get(new Tuple(50000)) === 100000, "Lookup 500 -> 1000");
    assert(map.get(new Tuple(99999)) === 199998, "Lookup 999 -> 1998");
});

In [ ]:
console.log('--- Test 6: Immutability Contract (Freeze on Hash) ---');
const mapContract = new RecursiveMap<RecursiveSet<number>, string>();
const setKey = new RecursiveSet<number>(1);

mapContract.set(setKey, "Value");
mapContract.hashCode;

let mutationThrew = false;
try {
    setKey.add(2);
} catch (e: any) {
    if (e.message && e.message.toLowerCase().includes("frozen")) {
        mutationThrew = true;
    }
}
assert(mutationThrew, "Key Set is frozen after Map hash calculation");

In [ ]:
console.log('--- Test 7: Edge Cases (Empty & Unicode) ---');
const edgeMap = new RecursiveMap<string, string>();
edgeMap.set("", "Empty");
edgeMap.set("ε", "Epsilon");
edgeMap.set("δ", "Delta");

assert(edgeMap.get("") === "Empty", "Empty String Key");
assert(edgeMap.get("ε") === "Epsilon", "Unicode Key (ε)");

edgeMap.delete("ε");
assert(edgeMap.get("ε") === undefined, "Delete worked");
assert(edgeMap.size === 2, "Size updated");

In [ ]:
console.log('--- Test 8: Overwrite Semantics ---');
const owMap = new RecursiveMap<Tuple<[number]>, string>();
const kInstance1 = new Tuple(1);
const kInstance2 = new Tuple(1);

owMap.set(kInstance1, "V1");
assert(owMap.size === 1, "Initial insert");

owMap.set(kInstance2, "V2"); // Overwrite
assert(owMap.size === 1, "Size remains 1");
assert(owMap.get(kInstance1) === "V2", "Value updated");

In [ ]:
console.log('--- Test 9: Delete & Reinsert (Hole Logic) ---');
const drMap = new RecursiveMap<string, number>();
drMap.set("k", 1);
assert(drMap.delete("k") === true, "Delete returned true");
assert(drMap.size === 0, "Map empty");
assert(drMap.get("k") === undefined, "Key gone");

drMap.set("k", 2);
assert(drMap.size === 1, "Reinserted");
assert(drMap.get("k") === 2, "New value correct");
console.log();

# The Big Bang

In [ ]:
const N_PARTICLES = 500_000; 
const N_COMPLEX = 50_000;

In [ ]:
console.log(`\n💥 INITIALIZING BIG BANG PROTOCOL 💥`);
console.log(`[Config] Particles: ${N_PARTICLES.toLocaleString()}`);
console.log(`[Config] Complex Structures: ${N_COMPLEX.toLocaleString()}`);

In [ ]:
console.log("🌌 PHASE 1: COSMIC INFLATION");

const universe = new RecursiveSet<number>();

measure(`Generating ${N_PARTICLES} unique particles`, () => {
    for (let i = 0; i < N_PARTICLES; i++) {
        universe.add(i);
    }
});

assert(universe.size === N_PARTICLES, "Universe size matches after inflation");

measure(`Scanning Universe (has check)`, () => {
    let found = 0;
    const limit = N_PARTICLES / 10;
    for (let i = 0; i < limit; i++) {
        const p = (i * 17) % N_PARTICLES; 
        if (universe.has(p)) found++;
    }
    assert(found === limit, "Lost particles in the void");
});

In [ ]:
console.log("⚛️  PHASE 2: MATTER FORMATION");

const reality = new RecursiveMap<Tuple<[number, number]>, RecursiveSet<number>>();

measure(`Forming ${N_COMPLEX} complex atoms`, () => {
    for (let i = 0; i < N_COMPLEX; i++) {
        const x = i % 1000;
        const y = (i * 31) % 1000;
        const key = new Tuple(x, y);

        const val = new RecursiveSet(i, i + 1);

        reality.set(key, val);
    }
});

console.log(`Reality Size: ${reality.size.toLocaleString()} dimensions created.`);

In [ ]:
console.log("🫰 PHASE 3: THE SNAP");

const snapUniverse = universe.clone();
const snapTarget = Math.floor(N_PARTICLES / 2);

measure(`Deleting ${snapTarget} particles (Evens)`, () => {
    for (let i = 0; i < N_PARTICLES; i += 2) {
        snapUniverse.remove(i);
    }
});

assert(snapUniverse.size === N_PARTICLES - Math.ceil(N_PARTICLES / 2), "Balance of universe is correct");

measure(`Verifying Survivors`, () => {
    for (let i = 1; i < 1000; i += 2) {
        if (!snapUniverse.has(i)) assert(false, `Survivor ${i} missing!`);
    }
});

In [ ]:
console.log("🕸️  PHASE 4: MULTIVERSE COLLISION");

const verseA = new RecursiveSet<Value>();
const verseB = new RecursiveSet<Value>();

const DEPTH = 500000;

measure(`Building Parallel Universes (${DEPTH} items)`, () => {
    for (let i = 0; i < DEPTH; i++) verseA.add(i);
    for (let i = DEPTH - 1; i >= 0; i--) verseB.add(i);
});

measure(`Comparing Universes (Equals)`, () => {
    const eq = verseA.equals(verseB);
    assert(eq, "Parallel universes are equal!");
});

measure(`Comparing Universes (HashCode)`, () => {
    const hA = verseA.hashCode;
    const hB = verseB.hashCode;
    assert(hA === hB, `Hash Match: ${hA} vs ${hB}`);
});

In [ ]:
console.log("❄️  PHASE 5: HEAT DEATH");

measure(`Total Annihilation (Loop Deletes)`, () => {
    for (let i = 0; i < N_PARTICLES; i++) {
        universe.remove(i);
    }
});

assert(universe.size === 0, "Universe is empty after heat death");
assert(universe.isEmpty(), "'isEmpty' verifies.");